# Robustness Experiments (Optional)

This notebook contains optional experiments to test the robustness of the federated PIDL model.

**These experiments are designed to run after the main experiments in notebook 01.**

---

**Experiments:**
1. Non-IID data heterogeneity sweep (Dirichlet α)
2. Client dropout simulation
3. PIDL diffusivity type comparison (Lorentzian vs Gaussian)
4. Feature layer ablation (layer2 vs layer3 vs layer4)
5. Partial data regime (reduced dataset size)

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/medical_fl_pidl'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)
os.environ['MEDICAL_FL_ROOT'] = PROJECT_ROOT

from configs.experiment_config import make_config
from utils.seed_utils import set_all_seeds
from utils.path_utils import get_results_dir, get_kaggle_cache_dir
from data.kaggle_loader import download_dataset, build_full_dataset_splits
from data.partitioning import partition_dataset
from federated.client_app import make_client_fn
from federated.server_app import build_server_app, run_simulation
from federated.task import resolve_device
from utils.logging_utils import make_run_dir, save_config_snapshot

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.grid': True})
print('Environment ready.')

## Helper: Run One Experiment

Convenience wrapper used by all robustness experiments below.

In [ ]:
def run_one_experiment(cfg, tag: str, cache_dir=None):
    """Download data, partition, run simulation, return final metrics."""
    from utils.logging_utils import save_metrics, load_metrics_jsonl
    from models.resnet_pidl import build_model, set_model_parameters
    from federated.task import evaluate_full
    from metrics.classification_metrics import compute_classification_metrics
    from metrics.calibration_metrics import compute_calibration_metrics
    from data.dataset_utils import build_val_transform
    from federated.client_app import _TransformedSubset
    from torch.utils.data import DataLoader

    device = resolve_device(cfg.device)
    set_all_seeds(cfg.seed)

    dataset_root = download_dataset(cfg.dataset, cache_dir=str(cache_dir) if cache_dir else None)
    splits = build_full_dataset_splits(cfg.dataset, dataset_root, seed=cfg.seed)
    partitions = partition_dataset(
        splits['train'], cfg.federated.num_clients,
        cfg.partitioning.strategy, cfg.partitioning.dirichlet_alpha, cfg.seed,
    )

    results_dir = get_results_dir()
    run_dir = make_run_dir(results_dir, cfg.experiment_name, run_name=tag)
    save_config_snapshot(cfg.to_dict(), run_dir)

    server_app, strategy = build_server_app(cfg, log_dir=run_dir)
    client_fn = make_client_fn(cfg, partitions, splits['val'], device)
    run_simulation(cfg, client_fn, server_app)

    # Evaluate final model on test set
    final_params = strategy.parameters
    model = build_model(cfg.dataset.num_classes, cfg.model).to(device)
    set_model_parameters(model, [p for p in final_params.tensors])

    val_transform = build_val_transform(cfg.dataset)
    test_ds = _TransformedSubset(splits['test'], val_transform)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

    eval_result = evaluate_full(model, test_loader, device, cfg.dataset.num_classes)
    cls_m = compute_classification_metrics(
        eval_result['all_probs'], eval_result['all_labels'],
        class_names=cfg.dataset.class_names, prefix='test_'
    )
    cal_m = compute_calibration_metrics(
        eval_result['all_probs'], eval_result['all_labels'], prefix='test_'
    )
    final_metrics = {**cls_m, **cal_m, 'test_loss': eval_result['loss']}

    save_metrics([final_metrics], run_dir, format='both', filename_stem='final_metrics')
    save_metrics(strategy.get_history(), run_dir, format='both', filename_stem='round_metrics')
    strategy.close()

    return final_metrics

## Experiment 1: Non-IID Heterogeneity Sweep

Varies Dirichlet α from 0.1 (highly heterogeneous) to 10 (near-IID).
Higher heterogeneity is expected to reduce accuracy and increase ECE.

In [ ]:
cache_dir = get_kaggle_cache_dir()
ALPHAS = [0.1, 0.3, 0.5, 1.0, 5.0, 10.0]

alpha_results = {}
for alpha in ALPHAS:
    print(f'\n>>> Dirichlet α = {alpha}')
    cfg = make_config(
        dataset_key='chest_xray',
        num_clients=3,
        num_rounds=15,
        lambda_pidl=0.01,
        partitioning='dirichlet',
        dirichlet_alpha=alpha,
        seed=42,
    )
    tag = f'robustness_dirichlet_alpha{alpha}'
    alpha_results[alpha] = run_one_experiment(cfg, tag, cache_dir)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Non-IID Heterogeneity Sweep (Dirichlet α)', fontsize=13)

alphas = list(alpha_results.keys())
accs = [alpha_results[a].get('test_accuracy', float('nan')) for a in alphas]
eces = [alpha_results[a].get('test_ece', float('nan')) for a in alphas]

axes[0].semilogx(alphas, [v*100 for v in accs], 'o-', color='steelblue', linewidth=2, markersize=7)
axes[0].set_xlabel('Dirichlet α')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Accuracy vs α')

axes[1].semilogx(alphas, eces, 'o-', color='tomato', linewidth=2, markersize=7)
axes[1].set_xlabel('Dirichlet α')
axes[1].set_ylabel('ECE')
axes[1].set_title('Calibration vs α')

plt.tight_layout()
plt.savefig(get_results_dir() / 'robustness_dirichlet_sweep.png', bbox_inches='tight', dpi=150)
plt.show()

## Experiment 2: PIDL Diffusivity Type Comparison

In [ ]:
from configs.experiment_config import make_config, PIDLConfig

diffusivity_results = {}
for diff_type in ['lorentzian', 'gaussian', 'none']:
    print(f'\n>>> Diffusivity type: {diff_type}')
    lambda_val = 0.01 if diff_type != 'none' else 0.0
    cfg = make_config(
        dataset_key='chest_xray',
        num_clients=3,
        num_rounds=15,
        lambda_pidl=lambda_val,
        seed=42,
    )
    if diff_type != 'none':
        cfg.pidl.diffusivity_type = diff_type
    tag = f'robustness_diffusivity_{diff_type}'
    diffusivity_results[diff_type] = run_one_experiment(cfg, tag, cache_dir)

print('\nDiffusivity comparison:')
for dtype, m in diffusivity_results.items():
    acc = m.get('test_accuracy', float('nan'))
    ece = m.get('test_ece', float('nan'))
    f1  = m.get('test_f1_macro', float('nan'))
    print(f'  {dtype:<12}: Acc={acc:.4f}  F1={f1:.4f}  ECE={ece:.4f}')

## Experiment 3: Feature Layer Ablation

Tests PIDL applied at different ResNet18 layers.

In [ ]:
layer_results = {}
for layer in ['layer2', 'layer3', 'layer4']:
    print(f'\n>>> PIDL on {layer}')
    cfg = make_config(
        dataset_key='chest_xray',
        num_clients=3,
        num_rounds=15,
        lambda_pidl=0.01,
        seed=42,
    )
    cfg.model.pidl_feature_layer = layer
    tag = f'robustness_layer_{layer}'
    layer_results[layer] = run_one_experiment(cfg, tag, cache_dir)

print('\nLayer ablation:')
for layer, m in layer_results.items():
    acc = m.get('test_accuracy', float('nan'))
    ece = m.get('test_ece', float('nan'))
    print(f'  {layer}: Acc={acc:.4f}  ECE={ece:.4f}')

## Experiment 4: Client Count Comparison (3 vs 4 vs 5)

In [ ]:
client_results = {}
for n_clients in [3, 4, 5]:
    print(f'\n>>> {n_clients} clients')
    cfg = make_config(
        dataset_key='chest_xray',
        num_clients=n_clients,
        num_rounds=15,
        lambda_pidl=0.01,
        seed=42,
    )
    tag = f'robustness_nclients_{n_clients}'
    client_results[n_clients] = run_one_experiment(cfg, tag, cache_dir)

print('\nClient count comparison:')
for n, m in client_results.items():
    acc = m.get('test_accuracy', float('nan'))
    ece = m.get('test_ece', float('nan'))
    print(f'  {n} clients: Acc={acc:.4f}  ECE={ece:.4f}')